# CUDA & Triton GPU Kernel Walkthrough

This notebook demonstrates custom GPU kernels for core deep learning operations.
All kernels have CPU-compatible reference implementations, so this runs anywhere.

**Topics covered:**
1. Fused softmax — why fusion reduces memory traffic
2. Tiled matrix multiplication — shared memory reuse
3. Flash Attention — online softmax for O(N) memory
4. Normalization kernels — LayerNorm and RMSNorm
5. Roofline analysis — identifying optimization targets

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import math
import time

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 1. Fused Softmax

Standard softmax requires 3 passes over the data: max (for stability), exp, and normalize.
A fused kernel does all three in one pass, reducing HBM traffic by ~3x.

The key insight: we can compute `max`, `exp(x - max)`, and `sum(exp)` in a single
streaming pass if we process the data in blocks and maintain running statistics.

In [ ]:
from triton_kernels.fused_softmax import fused_softmax, softmax_reference

torch.manual_seed(42)
x = torch.randn(8, 512)

result = fused_softmax(x)
expected = torch.softmax(x, dim=-1)

print(f'Input shape: {x.shape}')
print(f'Row sums (should be 1.0): {result.sum(dim=-1)[:4].tolist()}')
print(f'Max error vs torch.softmax: {(result - expected).abs().max().item():.2e}')

# Numerical stability test
x_large = torch.tensor([[1e6, 1e6 + 1, 1e6 + 2]])
print(f'\nLarge input softmax: {fused_softmax(x_large)}')
print('(No NaN/Inf — numerically stable)')

## 2. Tiled Matrix Multiplication

Matrix multiplication is compute-bound for large matrices but memory-bound for small ones.
Tiling loads blocks into fast shared memory (SRAM) so each element is reused `BLOCK_K` times
instead of being re-read from slow global memory (HBM) each time.

**Data reuse**: For tile size T, each element of A is used T times (once per column of the B tile),
and each element of B is used T times (once per row of the A tile).

In [ ]:
from triton_kernels.matmul import triton_matmul, matmul_reference

torch.manual_seed(42)
M, K, N = 128, 256, 64
a = torch.randn(M, K)
b = torch.randn(K, N)

# Compare different tile sizes
for block in [16, 32, 64]:
    result = matmul_reference(a, b, block_m=block, block_n=block, block_k=block)
    error = (result - (a @ b)).abs().max().item()
    print(f'Block size {block:>2}: max error = {error:.2e}')

# Arithmetic intensity calculation
flops = 2 * M * K * N
bytes_moved = (M * K + K * N + M * N) * 4  # float32
ai = flops / bytes_moved
print(f'\nArithmetic intensity: {ai:.1f} FLOP/byte')
print(f'Total FLOPs: {flops:,}')

## 3. Flash Attention

Standard attention materializes the full N×N attention matrix, using O(N²) memory.
Flash Attention tiles the computation and uses **online softmax** to avoid this.

The online softmax maintains running statistics `(m, ℓ)` — the current max and
normalizer — and rescales the accumulator when a new tile introduces a larger max.
This produces mathematically identical results with O(N) memory.

In [ ]:
from triton_kernels.flash_attention import flash_attention, _standard_attention

torch.manual_seed(42)
B, H, N, D = 2, 4, 128, 32
q = torch.randn(B, H, N, D)
k = torch.randn(B, H, N, D)
v = torch.randn(B, H, N, D)

# Flash attention (tiled, O(N) memory)
flash_out = flash_attention(q, k, v, block_q=32, block_kv=32)

# Standard attention (O(N²) memory)
std_out = _standard_attention(q, k, v)

print(f'Shape: ({B}, {H}, {N}, {D})')
print(f'Max error: {(flash_out - std_out).abs().max().item():.2e}')

# Memory comparison
attn_matrix_bytes = B * H * N * N * 4
output_bytes = B * H * N * D * 4
print(f'\nStandard attention matrix: {attn_matrix_bytes / 1024:.1f} KB')
print(f'Flash attention output only: {output_bytes / 1024:.1f} KB')
print(f'Memory savings: {attn_matrix_bytes / output_bytes:.1f}x')

## 4. Normalization Kernels

**LayerNorm** normalizes by mean and variance: `γ * (x - μ) / √(σ² + ε) + β`

**RMSNorm** simplifies by dropping the mean: `γ * x / √(mean(x²) + ε)`

RMSNorm is ~15% cheaper (one fewer reduction) and is used in LLaMA, Gemma, etc.

In [ ]:
from triton_kernels.layernorm import fused_layernorm
from triton_kernels.rmsnorm import fused_rmsnorm

torch.manual_seed(42)
x = torch.randn(16, 512)
gamma = torch.ones(512)
beta = torch.zeros(512)

# LayerNorm
ln_out = fused_layernorm(x, gamma, beta)
ln_expected = torch.nn.functional.layer_norm(x, [512])
print('LayerNorm:')
print(f'  Output mean: {ln_out.mean(dim=-1).abs().max().item():.2e} (should be ~0)')
print(f'  Output var:  {ln_out.var(dim=-1, unbiased=False).mean().item():.4f} (should be ~1)')
print(f'  Max error:   {(ln_out - ln_expected).abs().max().item():.2e}')

# RMSNorm
rms_out = fused_rmsnorm(x, gamma)
rms_manual = x / torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + 1e-6)
print('\nRMSNorm:')
print(f'  Output RMS:  {rms_out.pow(2).mean(dim=-1).sqrt().mean().item():.4f} (should be ~1)')
print(f'  Max error:   {(rms_out - rms_manual).abs().max().item():.2e}')

## 5. GELU Activation

GELU is the standard activation in Transformers. The tanh approximation avoids
the expensive `erf` function while being accurate to ~1e-4.

As a purely elementwise operation, GELU has arithmetic intensity ~1 FLOP/byte —
deeply memory-bound. Fusion with surrounding operations (e.g., the linear layer
before it) is the main optimization strategy.

In [ ]:
from triton_kernels.gelu import fused_gelu

torch.manual_seed(42)
x = torch.linspace(-4, 4, 100)

y_approx = fused_gelu(x, approximate=True)
y_exact = fused_gelu(x, approximate=False)
y_torch = torch.nn.functional.gelu(x, approximate='tanh')

print(f'Max error (approx vs torch): {(y_approx - y_torch).abs().max().item():.2e}')
print(f'Max error (approx vs exact): {(y_approx - y_exact).abs().max().item():.2e}')
print(f'\nGELU(0) = {fused_gelu(torch.tensor([0.0])).item():.4f}')
print(f'GELU(-2) = {fused_gelu(torch.tensor([-2.0])).item():.4f}')
print(f'GELU(2) = {fused_gelu(torch.tensor([2.0])).item():.4f}')

## 6. Roofline Analysis

The roofline model tells us whether a kernel is limited by compute or memory bandwidth.
Kernels below the ridge point are memory-bound; kernels above are compute-bound.

For each kernel, we compute:
- **Arithmetic intensity** = FLOPs / bytes transferred
- **Achieved performance** = FLOPs / wall-clock time
- **Roofline bound** = min(peak_compute, peak_bandwidth × AI)

In [ ]:
from benchmarks.roofline import run_roofline_analysis, print_roofline_table

hw, profiles = run_roofline_analysis()
print_roofline_table(hw, profiles)

print('\nKey takeaways:')
print('- GELU and Softmax are memory-bound (low AI) → fusion helps most')
print('- MatMul is compute-bound (high AI) → tensor cores / vectorization help')
print('- LayerNorm/RMSNorm are in between → both fusion and compute optimizations apply')

## Summary

| Kernel | Type | Arithmetic Intensity | Primary Optimization |
|--------|------|---------------------|---------------------|
| GELU | Elementwise | ~1 FLOP/byte | Fusion with linear layer |
| Softmax | Reduction + elementwise | ~2 FLOP/byte | Fused single-pass |
| LayerNorm | Reduction + affine | ~3 FLOP/byte | Welford one-pass + fusion |
| RMSNorm | Reduction + scale | ~2.5 FLOP/byte | Skip mean, one-pass |
| MatMul | Dense compute | ~N/3 FLOP/byte | Tiling, tensor cores |
| Attention | Mixed | Varies with N | Flash Attention (tiled online softmax) |

The roofline model guides optimization: memory-bound kernels benefit from fusion
and reduced HBM traffic, while compute-bound kernels benefit from hardware-specific
instructions (tensor cores, vectorized loads).